# Day 1 — Telco Customer Churn: Cleaning & EDA

This notebook:
1. Loads the raw Telco dataset
2. Cleans it carefully (especially `TotalCharges`)
3. Performs exploratory data analysis with real churn rates
4. Saves charts and the cleaned CSV

**How to run:** Open this file in VS Code or Jupyter, select the `venv` kernel, then run all cells.

## 1. Setup and load data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Paths work whether you run from project root or from notebooks/
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "telco_churn.csv"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_telco.csv"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)

print("Project root:", PROJECT_ROOT)
print("Raw data path:", RAW_PATH)
print("File exists:", RAW_PATH.exists())

In [ ]:
df_raw = pd.read_csv(RAW_PATH)
print("Raw shape:", df_raw.shape)
df_raw.head()

## 2. Missing-value analysis

In [ ]:
print("Null counts per column:")
print(df_raw.isnull().sum())

print("\nBlank string counts (common issue in this dataset):")
blank_counts = {}
for col in df_raw.columns:
    if df_raw[col].dtype == object:
        blank_counts[col] = (df_raw[col].astype(str).str.strip() == "").sum()
print(pd.Series(blank_counts).sort_values(ascending=False))

## 3. Duplicate analysis

In [ ]:
print("Duplicate rows:", df_raw.duplicated().sum())
print("Duplicate customerIDs:", df_raw["customerID"].duplicated().sum())

## 4. Data-type analysis

In [ ]:
print(df_raw.dtypes)
print("\nTotalCharges sample values (may include blanks):")
print(df_raw["TotalCharges"].head(10))
print("\nTotalCharges dtype before cleaning:", df_raw["TotalCharges"].dtype)

## 5. Clean the dataset

- Convert `TotalCharges` to numeric
- Handle blank/invalid values
- Keep useful rows; only drop rows that truly cannot be used

In [ ]:
df = df_raw.copy()

# Strip whitespace from object columns
for col in df.select_dtypes(include=["object", "string", "str"]).columns:
    df[col] = df[col].astype(str).str.strip()

# Convert TotalCharges to numeric (blank strings become NaN)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

invalid_total = df["TotalCharges"].isna().sum()
print(f"Rows with invalid/blank TotalCharges after conversion: {invalid_total}")

# Inspect those rows before deciding what to do
problem_rows = df[df["TotalCharges"].isna()]
print("\nProblem rows (tenure and charges):")
print(problem_rows[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]])

# These rows usually have tenure = 0 (brand-new customers with no bill yet).
# Filling TotalCharges with 0 is reasonable and keeps the customers in the dataset.
df.loc[df["TotalCharges"].isna() & (df["tenure"] == 0), "TotalCharges"] = 0.0

# If any invalid TotalCharges remain (unexpected), drop only those rows
remaining_invalid = df["TotalCharges"].isna().sum()
print(f"\nRemaining invalid TotalCharges after fill: {remaining_invalid}")
if remaining_invalid > 0:
    df = df.dropna(subset=["TotalCharges"]).copy()

# SeniorCitizen is already 0/1 numeric — keep as int
df["SeniorCitizen"] = df["SeniorCitizen"].astype(int)

print("\nCleaned shape:", df.shape)
print("Nulls after cleaning:")
print(df.isnull().sum())
print("\nTotalCharges dtype after cleaning:", df["TotalCharges"].dtype)
df.head()

## 6. Verify cleaned dataset

In [ ]:
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customerIDs:", df["customerID"].duplicated().sum())
print("\nData types:")
print(df.dtypes)
print("\nDescribe:")
df.describe()

## 7. Churn rate calculations (real numbers from data)

In [ ]:
def churn_rate_table(data: pd.DataFrame, column: str) -> pd.DataFrame:
    """Return counts and churn % for each category in a column."""
    table = (
        data.groupby(column)["Churn"]
        .agg(
            total_customers="count",
            churned=lambda s: (s == "Yes").sum(),
        )
        .reset_index()
    )
    table["churn_rate_pct"] = (table["churned"] / table["total_customers"] * 100).round(2)
    return table.sort_values("churn_rate_pct", ascending=False)


overall_churn = (df["Churn"] == "Yes").mean() * 100
print(f"Overall churn rate: {overall_churn:.2f}%")
print(f"Churned customers: {(df['Churn'] == 'Yes').sum()}")
print(f"Non-churned customers: {(df['Churn'] == 'No').sum()}")

print("\n--- Churn rate by Contract ---")
print(churn_rate_table(df, "Contract"))

print("\n--- Churn rate by InternetService ---")
print(churn_rate_table(df, "InternetService"))

print("\n--- Churn rate by PaymentMethod ---")
print(churn_rate_table(df, "PaymentMethod"))

## 8. EDA visualizations

Charts are saved to `reports/figures/`.

In [ ]:
def save_fig(name: str) -> None:
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"Saved: {path}")


# 1. Churn distribution
plt.figure()
ax = sns.countplot(data=df, x="Churn", hue="Churn", palette="Set2", legend=False)
ax.set_title("1. Churn Distribution")
ax.set_xlabel("Churn")
ax.set_ylabel("Number of Customers")
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
save_fig("01_churn_distribution.png")
plt.show()

In [ ]:
# 2. Contract vs Churn
plt.figure(figsize=(9, 5))
ax = sns.countplot(data=df, x="Contract", hue="Churn", palette="Set2")
ax.set_title("2. Contract vs Churn")
ax.set_ylabel("Number of Customers")
save_fig("02_contract_vs_churn.png")
plt.show()

print(churn_rate_table(df, "Contract"))

In [ ]:
# 3. Tenure vs Churn
plt.figure()
ax = sns.histplot(data=df, x="tenure", hue="Churn", bins=30, kde=False, multiple="stack", palette="Set2")
ax.set_title("3. Tenure vs Churn")
ax.set_xlabel("Tenure (months)")
ax.set_ylabel("Number of Customers")
save_fig("03_tenure_vs_churn.png")
plt.show()

print("Average tenure by Churn:")
print(df.groupby("Churn")["tenure"].mean().round(2))

In [ ]:
# 4. MonthlyCharges vs Churn
plt.figure()
ax = sns.boxplot(data=df, x="Churn", y="MonthlyCharges", hue="Churn", palette="Set2", legend=False)
ax.set_title("4. MonthlyCharges vs Churn")
save_fig("04_monthlycharges_vs_churn.png")
plt.show()

print("Average MonthlyCharges by Churn:")
print(df.groupby("Churn")["MonthlyCharges"].mean().round(2))

In [ ]:
# 5. InternetService vs Churn
plt.figure(figsize=(9, 5))
ax = sns.countplot(data=df, x="InternetService", hue="Churn", palette="Set2")
ax.set_title("5. InternetService vs Churn")
ax.set_ylabel("Number of Customers")
save_fig("05_internetservice_vs_churn.png")
plt.show()

print(churn_rate_table(df, "InternetService"))

In [ ]:
# 6. PaymentMethod vs Churn
plt.figure(figsize=(11, 5))
ax = sns.countplot(data=df, x="PaymentMethod", hue="Churn", palette="Set2")
ax.set_title("6. PaymentMethod vs Churn")
ax.set_ylabel("Number of Customers")
plt.xticks(rotation=20, ha="right")
save_fig("06_paymentmethod_vs_churn.png")
plt.show()

print(churn_rate_table(df, "PaymentMethod"))

In [ ]:
# 7. PaperlessBilling vs Churn
plt.figure()
ax = sns.countplot(data=df, x="PaperlessBilling", hue="Churn", palette="Set2")
ax.set_title("7. PaperlessBilling vs Churn")
ax.set_ylabel("Number of Customers")
save_fig("07_paperlessbilling_vs_churn.png")
plt.show()

print(churn_rate_table(df, "PaperlessBilling"))

In [ ]:
# 8. SeniorCitizen vs Churn
plt.figure()
plot_df = df.copy()
plot_df["SeniorCitizen_label"] = plot_df["SeniorCitizen"].map({0: "Not Senior", 1: "Senior"})
ax = sns.countplot(data=plot_df, x="SeniorCitizen_label", hue="Churn", palette="Set2")
ax.set_title("8. SeniorCitizen vs Churn")
ax.set_xlabel("SeniorCitizen")
ax.set_ylabel("Number of Customers")
save_fig("08_seniorcitizen_vs_churn.png")
plt.show()

print(churn_rate_table(df, "SeniorCitizen"))

In [ ]:
# 9. Partner vs Churn
plt.figure()
ax = sns.countplot(data=df, x="Partner", hue="Churn", palette="Set2")
ax.set_title("9. Partner vs Churn")
ax.set_ylabel("Number of Customers")
save_fig("09_partner_vs_churn.png")
plt.show()

print(churn_rate_table(df, "Partner"))

In [ ]:
# 10. Dependents vs Churn
plt.figure()
ax = sns.countplot(data=df, x="Dependents", hue="Churn", palette="Set2")
ax.set_title("10. Dependents vs Churn")
ax.set_ylabel("Number of Customers")
save_fig("10_dependents_vs_churn.png")
plt.show()

print(churn_rate_table(df, "Dependents"))

## 9. Business insights

Run the cell below **after** the charts/tables above. It writes insights based only on the actual numbers in your cleaned data.

In [ ]:
overall = (df["Churn"] == "Yes").mean() * 100
contract = churn_rate_table(df, "Contract")
internet = churn_rate_table(df, "InternetService")
payment = churn_rate_table(df, "PaymentMethod")
paperless = churn_rate_table(df, "PaperlessBilling")
senior = churn_rate_table(df, "SeniorCitizen")
partner = churn_rate_table(df, "Partner")
dependents = churn_rate_table(df, "Dependents")

avg_tenure = df.groupby("Churn")["tenure"].mean().round(2)
avg_monthly = df.groupby("Churn")["MonthlyCharges"].mean().round(2)

print("BUSINESS INSIGHTS (from this dataset only)\n")
print(f"1. Overall churn rate is {overall:.2f}%.")
print(
    f"2. Highest contract churn: {contract.iloc[0]['Contract']} "
    f"({contract.iloc[0]['churn_rate_pct']}%). "
    f"Lowest: {contract.iloc[-1]['Contract']} ({contract.iloc[-1]['churn_rate_pct']}%)."
)
print(
    f"3. Highest internet-service churn: {internet.iloc[0]['InternetService']} "
    f"({internet.iloc[0]['churn_rate_pct']}%)."
)
print(
    f"4. Highest payment-method churn: {payment.iloc[0]['PaymentMethod']} "
    f"({payment.iloc[0]['churn_rate_pct']}%)."
)
print(
    f"5. Average tenure — Churned: {avg_tenure.get('Yes', float('nan'))} months, "
    f"Not churned: {avg_tenure.get('No', float('nan'))} months."
)
print(
    f"6. Average monthly charges — Churned: ${avg_monthly.get('Yes', float('nan'))}, "
    f"Not churned: ${avg_monthly.get('No', float('nan'))}."
)
print(
    f"7. Paperless billing churn rates: "
    + ", ".join(f"{r.PaperlessBilling}={r.churn_rate_pct}%" for r in paperless.itertuples())
)
print(
    f"8. SeniorCitizen churn rates: "
    + ", ".join(f"{r.SeniorCitizen}={r.churn_rate_pct}%" for r in senior.itertuples())
)
print(
    f"9. Partner churn rates: "
    + ", ".join(f"{r.Partner}={r.churn_rate_pct}%" for r in partner.itertuples())
)
print(
    f"10. Dependents churn rates: "
    + ", ".join(f"{r.Dependents}={r.churn_rate_pct}%" for r in dependents.itertuples())
)

## 10. Save cleaned dataset and verify reload

In [ ]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print(f"Saved cleaned dataset to: {PROCESSED_PATH}")
print(f"File exists: {PROCESSED_PATH.exists()}")

# Verify we can load it again
df_check = pd.read_csv(PROCESSED_PATH)
print(f"Reloaded shape: {df_check.shape}")
print(df_check.head())
assert df_check.shape[0] == df.shape[0], "Row count mismatch after reload"
assert df_check.shape[1] == df.shape[1], "Column count mismatch after reload"
print("Verification passed.")